# Apache Spark Basics - Introduction to PySpark

This notebook introduces the fundamentals of Apache Spark using PySpark. You'll learn about:
- SparkSession and SparkContext
- RDDs (Resilient Distributed Datasets)
- DataFrames and basic operations
- Reading and writing data

## Prerequisites
Make sure your Spark cluster is running via Docker Compose.

## 1. Setting up Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd

# Create Spark Session
spark = SparkSession.builder \
    .appName("Spark Basics Demo") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Show Spark version and basic info
print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")
print(f"Application Name: {spark.sparkContext.appName}")

## 2. Working with RDDs (Resilient Distributed Datasets)

In [ ]:
# Create an RDD from a Python list
numbers = spark.sparkContext.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Basic RDD operations
print("Original RDD:", numbers.collect())
print("Count:", numbers.count())
print("First element:", numbers.first())
print("Take 3 elements:", numbers.take(3))

In [ ]:
# RDD transformations (lazy evaluation)
squared = numbers.map(lambda x: x ** 2)
evens = numbers.filter(lambda x: x % 2 == 0)
doubled = numbers.map(lambda x: x * 2)

print("Squared:", squared.collect())
print("Even numbers:", evens.collect())
print("Doubled:", doubled.collect())

In [ ]:
# RDD actions
total_sum = numbers.reduce(lambda a, b: a + b)
max_value = numbers.max()
min_value = numbers.min()

print(f"Sum: {total_sum}")
print(f"Max: {max_value}")
print(f"Min: {min_value}")

## 3. Working with DataFrames

In [ ]:
# Create a DataFrame from Python data
data = [
    ("Alice", 25, "Engineer"),
    ("Bob", 30, "Data Scientist"),
    ("Charlie", 35, "Manager"),
    ("Diana", 28, "Analyst")
]

schema = ["name", "age", "job"]
df = spark.createDataFrame(data, schema)

# Show DataFrame
df.show()
df.printSchema()

In [ ]:
# Basic DataFrame operations
print("Row count:", df.count())
print("Columns:", df.columns)

# Select specific columns
df.select("name", "age").show()

# Filter data
df.filter(df.age > 28).show()

# Add a new column
df.withColumn("age_plus_10", df.age + 10).show()

## 4. Reading Real Data - Sales Dataset

In [ ]:
# Read the sales data CSV file
sales_df = spark.read.csv("/home/jovyan/data/csv/sales_data.csv", header=True, inferSchema=True)

# Display basic information
print(f"Number of records: {sales_df.count()}")
print(f"Number of columns: {len(sales_df.columns)}")

sales_df.printSchema()

In [ ]:
# Show first few rows
sales_df.show(5)

# Show summary statistics
sales_df.describe().show()

## 5. Basic Data Analysis with Spark SQL

In [ ]:
# Create a temporary view for SQL queries
sales_df.createOrReplaceTempView("sales")

# Query 1: Total sales by category
result1 = spark.sql("""
    SELECT category, 
           COUNT(*) as transaction_count,
           ROUND(SUM(total_amount), 2) as total_sales
    FROM sales 
    GROUP BY category 
    ORDER BY total_sales DESC
""")

result1.show()

In [ ]:
# Query 2: Average order value by region
result2 = spark.sql("""
    SELECT region,
           COUNT(*) as orders,
           ROUND(AVG(total_amount), 2) as avg_order_value,
           ROUND(SUM(total_amount), 2) as total_revenue
    FROM sales
    GROUP BY region
    ORDER BY total_revenue DESC
""")

result2.show()

## 6. DataFrame API Operations

In [ ]:
# Same analysis using DataFrame API instead of SQL
# Group by category and calculate aggregations
category_analysis = sales_df.groupBy("category") \
    .agg(
        count("*").alias("transaction_count"),
        round(sum("total_amount"), 2).alias("total_sales"),
        round(avg("total_amount"), 2).alias("avg_order_value")
    ) \
    .orderBy(desc("total_sales"))

category_analysis.show()

In [ ]:
# Filter and transform data
high_value_sales = sales_df \
    .filter(col("total_amount") > 1000) \
    .select(
        "transaction_id",
        "date",
        "category",
        "total_amount",
        "region"
    ) \
    .orderBy(desc("total_amount"))

print(f"High value transactions (>$1000): {high_value_sales.count()}")
high_value_sales.show(10)

## 7. Writing Data

In [ ]:
# Save the category analysis as Parquet
category_analysis.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/parquet/category_analysis")

print("Category analysis saved as Parquet")

# Save high-value sales as CSV
high_value_sales.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/home/jovyan/data/csv/high_value_sales")

print("High-value sales saved as CSV")

## 8. Performance Monitoring

In [ ]:
# Check execution plan
print("Execution plan for category analysis:")
category_analysis.explain(True)

In [ ]:
# Cache frequently used DataFrames
sales_df.cache()
print(f"Sales DataFrame cached. Storage level: {sales_df.storageLevel}")

# Check cache status
print(f"Is cached: {sales_df.is_cached}")

## 9. Cleanup

In [ ]:
# Unpersist cached DataFrames
sales_df.unpersist()

# Stop Spark session (optional - usually done at end of application)
# spark.stop()

print("Cleanup completed!")

## Summary

In this notebook, you learned:
1. **SparkSession**: The entry point for Spark functionality
2. **RDDs**: Low-level distributed collections with transformations and actions
3. **DataFrames**: Higher-level API with schema and optimizations
4. **Spark SQL**: SQL interface for structured data processing
5. **Data I/O**: Reading from and writing to various formats
6. **Performance**: Caching and execution plan analysis

## Next Steps
- Try the advanced analytics notebook for more complex operations
- Explore streaming data processing
- Learn about machine learning with Spark MLlib